# Vision-Based Landslide Forecasting: Image Acquisition
This notebook connects to Google Earth Engine and downloads 224x224 pixel Sentinel-2 satellite images for the landslide locations (positive samples) and non-landslide locations (negative samples).

In [1]:
import ee
import os
import requests
import random
import geopandas as gpd
import warnings
from IPython.display import Image, display
warnings.filterwarnings('ignore')

## 1. Initialize Earth Engine
Make sure you have run `earthengine authenticate` in the terminal first!

In [3]:
# Initialize with the project you created
ee.Authenticate()
ee.Initialize(project='vision-based-landslide-506219')

In [4]:
import ee
import os
import requests
import random
import geopandas as gpd
import warnings
from IPython.display import Image, display
warnings.filterwarnings('ignore')

## 2. Load Coordinates & Generate Negative Samples
We read the shapefile again to get our positive coordinates.

In [5]:
shapefile_path = r"C:\Art_of_Coding\visionbased_grp\-Vision-Based-Landslide-Forecasting-Using-Satellite-Imagery\Landslides\SHP"
gdf = gpd.read_file(shapefile_path)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")
    
gdf['centroid'] = gdf.geometry.centroid

# Create lists for coordinates
positive_coords = []
for pt in gdf['centroid']:
    positive_coords.append((pt.x, pt.y)) # (Longitude, Latitude)

print(f"Loaded {len(positive_coords)} positive landslide coordinates.")

# Generate Negative Samples (approx 2-3km away in a random direction)
# 1 degree is roughly 111km, so 2.5km is roughly 0.022 degrees
negative_coords = []
for lon, lat in positive_coords:
    # Randomly shift by a safe distance
    lat_shift = random.choice([1, -1]) * random.uniform(0.02, 0.04)
    lon_shift = random.choice([1, -1]) * random.uniform(0.02, 0.04)
    negative_coords.append((lon + lon_shift, lat + lat_shift))
    
print(f"Generated {len(negative_coords)} negative non-landslide coordinates.")

Loaded 4225 positive landslide coordinates.
Generated 4225 negative non-landslide coordinates.


## 3. The Download Function
This function downloads a 224x224 PNG image centered around a given coordinate.

In [7]:
def download_sentinel_image(lon, lat, filename, date_start='2025-12-05', date_end='2026-04-30'):
    try:
        # 1. Define the point and a 1120m buffer (~2240m width/height)
        # At 10m/pixel resolution, 2240m = 224 pixels.
        point = ee.Geometry.Point([lon, lat])
        region = point.buffer(1120).bounds()
        
        # 2. Filter Sentinel-2 Surface Reflectance Image Collection
        collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
            .filterBounds(region) \
            .filterDate(date_start, date_end) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
            .sort('CLOUDY_PIXEL_PERCENTAGE') # Get least cloudy image first
            
        # 3. Check if image exists
        if collection.size().getInfo() == 0:
            print(f"No cloud-free images found for {filename}")
            return False
            
        # 4. Extract True Color (RGB) bands
        image = collection.first().select(['B4', 'B3', 'B2'])
        
        # 5. Visualize (convert to 8-bit image for PNG)
        vis_image = image.visualize(min=0, max=3000, bands=['B4', 'B3', 'B2'])
        
        # 6. Get download URL
        url = vis_image.getThumbURL({
            'region': region,
            'dimensions': '224x224',
            'format': 'png'
        })
        
        # 7. Save to disk
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
        return True
        
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return False

## 4. Let's Test It! (Download 5 images)
Before we download all 8000+ images, let's download a small sample to make sure everything works perfectly.

In [8]:
# Create folders for the dataset
os.makedirs('dataset/positive', exist_ok=True)
os.makedirs('dataset/negative', exist_ok=True)

print("Downloading 5 positive (landslide) samples...")
for i in range(5):
    lon, lat = positive_coords[i]
    filename = f'dataset/positive/landslide_{i}.png'
    download_sentinel_image(lon, lat, filename)
    print(f"Saved {filename}")

print("\nDownloading 5 negative (non-landslide) samples...")
for i in range(5):
    lon, lat = negative_coords[i]
    filename = f'dataset/negative/non_landslide_{i}.png'
    download_sentinel_image(lon, lat, filename)
    print(f"Saved {filename}")

print("\nDone! Open the 'dataset' folder to view the images.")

Saved dataset/positive/landslide_0.png
Saved dataset/positive/landslide_1.png
Saved dataset/positive/landslide_2.png
Saved dataset/positive/landslide_3.png
Saved dataset/positive/landslide_4.png

Saved dataset/negative/non_landslide_0.png
Saved dataset/negative/non_landslide_1.png
Saved dataset/negative/non_landslide_2.png
Saved dataset/negative/non_landslide_3.png
Saved dataset/negative/non_landslide_4.png

Done! Open the 'dataset' folder to view the images.
